In [2]:
import pandas as pd
import numpy as np
import requests
import time
import random
from io import StringIO

## Players Stats

In [ ]:
def scrape_per_game_stats(year: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    url = f"https://www.basketball-reference.com/leagues/NBA_{year}_per_game.html"

    print(f"正在抓取 {year} 賽季 per_game 數據...")
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        tables = pd.read_html(StringIO(response.text))
        
        # 第一個表格是 Regular Season
        df_reg = tables[0]
        df_reg = df_reg[df_reg['Player'] != 'Player'].copy() # 清理重複標題
        df_reg['year'] = year
        df_reg['Type'] = 'Regular'
        df_reg.drop(columns=['Rk','Awards','GS'], inplace=True) # 刪除排名欄位
        
        # 第二個表格是 Playoffs
        df_playoffs = pd.DataFrame()
        if len(tables) > 1:
            for t in tables[1:]:
                # 檢查這個表格的欄位是不是我們要的球員數據表，且有沒有包含季後賽關鍵字
                if 'Player' in t.columns and t['Player'].iloc[0] != 'Player':
                    df_playoffs = t.copy()
                    df_playoffs = df_playoffs[df_playoffs['Player'] != 'Player']
                    df_playoffs['year'] = year
                    df_playoffs['Type'] = 'Playoffs'
                    df_playoffs.drop(columns=['Rk','Awards','GS'], inplace=True) # 刪除排名、先發、獎項欄位
                    break
                    
        return df_reg, df_playoffs
    else:
        print(f"錯誤：無法存取 {year} 數據。")
        return pd.DataFrame(), pd.DataFrame()
    
def scrape_advanced_stats(year: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    url = f"https://www.basketball-reference.com/leagues/NBA_{year}_advanced.html"

    print(f"正在抓取 {year} 賽季 advanced 數據...")
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        tables = pd.read_html(StringIO(response.text))
        
        # 第一個表格是 Regular Season
        df_reg = tables[0]
        df_reg = df_reg[df_reg['Player'] != 'Player'].copy() # 清理重複標題
        df_reg['year'] = year
        df_reg['Type'] = 'Regular'
        df_reg.drop(columns=['Rk','Awards','GS'], inplace=True) # 刪除排名欄位
        # 第二個表格是 Playoffs
        df_playoffs = pd.DataFrame()
        if len(tables) > 1:
            for t in tables[1:]:
                # 檢查這個表格的欄位是不是我們要的球員數據表，且有沒有包含季後賽關鍵字
                if 'Player' in t.columns and t['Player'].iloc[0] != 'Player':
                    df_playoffs = t.copy()
                    df_playoffs = df_playoffs[df_playoffs['Player'] != 'Player']
                    df_playoffs['year'] = year
                    df_playoffs['Type'] = 'Playoffs'
                    df_playoffs.drop(columns=['Rk','Awards','GS'], inplace=True) # 刪除排名、先發、獎項欄位
                    break
                    
        return df_reg, df_playoffs
    else:
        print(f"錯誤：無法存取 {year} 數據。")
        return pd.DataFrame(), pd.DataFrame()

In [27]:
years = list(range(2014, 2025))

all_reg_data = []
all_playoffs_data = []

for year in years:
    # 1. 抓取當年度數據 (函數回傳 df_reg, df_playoffs)
    df_per_reg, df_per_playoffs = scrape_per_game_stats(year)
    df_adv_reg, df_adv_playoffs = scrape_advanced_stats(year)
    
    # 2. 處理常規賽合併
    if not df_per_reg.empty and not df_adv_reg.empty:
        # 避開重複欄位 (如 Pos, Age, G 等兩張表都有的欄位)，只留 advanced 特有的欄位
        adv_cols_to_keep = ['Player', 'Team'] + [c for c in df_adv_reg.columns if c not in df_per_reg.columns]
        
        # 使用 Player 和 Team (球隊) 作為主鍵合併
        reg_merged = pd.merge(df_per_reg, df_adv_reg[adv_cols_to_keep], on=['Player', 'Team'], how='left')
        all_reg_data.append(reg_merged)

    # 3. 處理季後賽合併
    if not df_per_playoffs.empty and not df_adv_playoffs.empty:
        adv_cols_to_keep_po = ['Player', 'Team'] + [c for c in df_adv_playoffs.columns if c not in df_per_playoffs.columns]
        po_merged = pd.merge(df_per_playoffs, df_adv_playoffs[adv_cols_to_keep_po], on=['Player', 'Team'], how='left')
        all_playoffs_data.append(po_merged)
        
    # 隨機暫停
    time.sleep(random.uniform(2, 4))

正在抓取 2014 賽季 per_game 數據...
正在抓取 2014 賽季 advanced 數據...
正在抓取 2015 賽季 per_game 數據...
正在抓取 2015 賽季 advanced 數據...
正在抓取 2016 賽季 per_game 數據...
正在抓取 2016 賽季 advanced 數據...
正在抓取 2017 賽季 per_game 數據...
正在抓取 2017 賽季 advanced 數據...
正在抓取 2018 賽季 per_game 數據...
正在抓取 2018 賽季 advanced 數據...
正在抓取 2019 賽季 per_game 數據...
正在抓取 2019 賽季 advanced 數據...
正在抓取 2020 賽季 per_game 數據...
正在抓取 2020 賽季 advanced 數據...
正在抓取 2021 賽季 per_game 數據...
正在抓取 2021 賽季 advanced 數據...
正在抓取 2022 賽季 per_game 數據...
正在抓取 2022 賽季 advanced 數據...
正在抓取 2023 賽季 per_game 數據...
正在抓取 2023 賽季 advanced 數據...
正在抓取 2024 賽季 per_game 數據...
正在抓取 2024 賽季 advanced 數據...


In [ ]:
# 4. 最終垂直組合 11 年的資料
final_reg_df = pd.concat(all_reg_data, ignore_index=True)
final_playoffs_df = pd.concat(all_playoffs_data, ignore_index=True)

print("常規賽資料維度:", final_reg_df.shape)
print("季後賽資料維度:", final_playoffs_df.shape)

常規賽資料維度: (7400, 51)
季後賽資料維度: (2379, 51)


In [ ]:
# 接著就可以直接 to_csv 存檔了
final_reg_df.to_csv('../../../data/raw/Stats_reg_2014_2024.csv', index=False)
final_playoffs_df.to_csv('../../../data/raw/Stats_playoffs_2014_2024.csv', index=False)

### 爬取歷史球員數據: 1980 - 2025

In [ ]:
import pandas as pd
import requests
from io import StringIO
import time
import random

def scrape_per_game_stats(year: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    url = f"https://www.basketball-reference.com/leagues/NBA_{year}_per_game.html"

    print(f"正在抓取 {year} 賽季 per_game 數據...")
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        tables = pd.read_html(StringIO(response.text))
        
        # 第一個表格是 Regular Season
        df_reg = tables[0]
        df_reg = df_reg[df_reg['Player'] != 'Player'].copy() # 清理重複標題
        df_reg['year'] = year
        df_reg['Type'] = 'Regular'
        # 使用 errors='ignore'，如果沒有該欄位就不會報錯
        df_reg.drop(columns=['Rk', 'Awards', 'GS'], errors='ignore', inplace=True) 
        
        # 第二個表格是 Playoffs
        df_playoffs = pd.DataFrame()
        if len(tables) > 1:
            for t in tables[1:]:
                if 'Player' in t.columns and t['Player'].iloc[0] != 'Player':
                    df_playoffs = t.copy()
                    df_playoffs = df_playoffs[df_playoffs['Player'] != 'Player']
                    df_playoffs['year'] = year
                    df_playoffs['Type'] = 'Playoffs'
                    # 同樣使用 errors='ignore'
                    df_playoffs.drop(columns=['Rk', 'Awards', 'GS'], errors='ignore', inplace=True) 
                    break
                    
        return df_reg, df_playoffs
    else:
        print(f"錯誤：無法存取 {year} per_game 數據。狀態碼: {response.status_code}")
        return pd.DataFrame(), pd.DataFrame()
    
def scrape_advanced_stats(year: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    url = f"https://www.basketball-reference.com/leagues/NBA_{year}_advanced.html"

    print(f"正在抓取 {year} 賽季 advanced 數據...")
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        tables = pd.read_html(StringIO(response.text))
        
        # 第一個表格是 Regular Season
        df_reg = tables[0]
        df_reg = df_reg[df_reg['Player'] != 'Player'].copy() 
        df_reg['year'] = year
        df_reg['Type'] = 'Regular'
        df_reg.drop(columns=['Rk', 'Awards', 'GS'], errors='ignore', inplace=True) 
        
        # 第二個表格是 Playoffs
        df_playoffs = pd.DataFrame()
        if len(tables) > 1:
            for t in tables[1:]:
                if 'Player' in t.columns and t['Player'].iloc[0] != 'Player':
                    df_playoffs = t.copy()
                    df_playoffs = df_playoffs[df_playoffs['Player'] != 'Player']
                    df_playoffs['year'] = year
                    df_playoffs['Type'] = 'Playoffs'
                    df_playoffs.drop(columns=['Rk', 'Awards', 'GS'], errors='ignore', inplace=True) 
                    break
                    
        return df_reg, df_playoffs
    else:
        print(f"錯誤：無法存取 {year} advanced 數據。狀態碼: {response.status_code}")
        return pd.DataFrame(), pd.DataFrame()

if __name__ == "__main__":
    years = list(range(1980, 2026))

    all_reg_data = []
    all_playoffs_data = []

    for year in years:
        df_per_reg, df_per_playoffs = scrape_per_game_stats(year)
        # 爬取進階數據前也稍微暫停一下，降低被 ban 的風險
        time.sleep(random.uniform(1, 2))
        df_adv_reg, df_adv_playoffs = scrape_advanced_stats(year)
        
        # 2. 處理常規賽合併
        if not df_per_reg.empty and not df_adv_reg.empty:
            adv_cols_to_keep = ['Player', 'Team'] + [c for c in df_adv_reg.columns if c not in df_per_reg.columns]
            reg_merged = pd.merge(df_per_reg, df_adv_reg[adv_cols_to_keep], on=['Player', 'Team'], how='left')
            all_reg_data.append(reg_merged)

        # 3. 處理季後賽合併
        if not df_per_playoffs.empty and not df_adv_playoffs.empty:
            adv_cols_to_keep_po = ['Player', 'Team'] + [c for c in df_adv_playoffs.columns if c not in df_per_playoffs.columns]
            po_merged = pd.merge(df_per_playoffs, df_adv_playoffs[adv_cols_to_keep_po], on=['Player', 'Team'], how='left')
            all_playoffs_data.append(po_merged)
            
        # 隨機暫停 (每個賽季間隔)
        time.sleep(random.uniform(3, 6))


正在抓取 1980 賽季 per_game 數據...
正在抓取 1980 賽季 advanced 數據...
正在抓取 1981 賽季 per_game 數據...
正在抓取 1981 賽季 advanced 數據...
正在抓取 1982 賽季 per_game 數據...
正在抓取 1982 賽季 advanced 數據...
正在抓取 1983 賽季 per_game 數據...
正在抓取 1983 賽季 advanced 數據...
正在抓取 1984 賽季 per_game 數據...
正在抓取 1984 賽季 advanced 數據...
正在抓取 1985 賽季 per_game 數據...
正在抓取 1985 賽季 advanced 數據...
正在抓取 1986 賽季 per_game 數據...
正在抓取 1986 賽季 advanced 數據...
正在抓取 1987 賽季 per_game 數據...
正在抓取 1987 賽季 advanced 數據...
正在抓取 1988 賽季 per_game 數據...
正在抓取 1988 賽季 advanced 數據...
正在抓取 1989 賽季 per_game 數據...
正在抓取 1989 賽季 advanced 數據...
正在抓取 1990 賽季 per_game 數據...
正在抓取 1990 賽季 advanced 數據...
正在抓取 1991 賽季 per_game 數據...
正在抓取 1991 賽季 advanced 數據...
正在抓取 1992 賽季 per_game 數據...
正在抓取 1992 賽季 advanced 數據...
正在抓取 1993 賽季 per_game 數據...
正在抓取 1993 賽季 advanced 數據...
正在抓取 1994 賽季 per_game 數據...
正在抓取 1994 賽季 advanced 數據...
正在抓取 1995 賽季 per_game 數據...
正在抓取 1995 賽季 advanced 數據...
正在抓取 1996 賽季 per_game 數據...
正在抓取 1996 賽季 advanced 數據...
正在抓取 1997 賽季 per_game 數據...
正在抓取 1997 賽季 advance

OSError: Cannot save file into a non-existent directory: '..\..\data\raw'

In [6]:
# 合併所有年份並存出 CSV
if all_reg_data:
    final_reg = pd.concat(all_reg_data, ignore_index=True)
    final_reg.to_csv('../../../data/raw/history_stats_reg.csv', index=False)
    print(f"✅ 歷史例行賽數據已儲存，共 {len(final_reg)} 筆")
    
if all_playoffs_data:
    final_playoffs = pd.concat(all_playoffs_data, ignore_index=True)
    final_playoffs.to_csv('../../../data/raw/history_stats_playoff.csv', index=False)
    print(f"✅ 歷史季後賽數據已儲存，共 {len(final_playoffs)} 筆")

✅ 歷史例行賽數據已儲存，共 24664 筆
✅ 歷史季後賽數據已儲存，共 8863 筆
